# Credit Card PDFs → Dated Folder, Summary, Dashboard, Suggestions

**Repeatable pipeline.** Drop one or more Wealthsimple credit-card activity PDFs into a folder, run all cells,
and get:

1. A new folder named after the date range covered by the statements (e.g. `2026-05-02_to_2026-05-14`)
2. `transactions.csv` and `summary.md` — clean data + a written-up overview
3. `spending_dashboard.html` — the same interactive dashboard style as the existing one in this project
4. `suggestions.html` — a separate page with **auto-generated suggestions** to lower spend

## How to use

1. Put your PDFs anywhere (default: a `PDFs/` folder next to this notebook).
2. Edit `INPUT_DIR` and `OUTPUT_BASE_DIR` in cell 2 if needed.
3. `Cell → Run All`.

## Prerequisites

```bash
pip install pdfplumber pandas
```

That's it — no OCR, no ffmpeg. PDFs already have embedded text.

## 1. Configuration

In [ ]:
# === EDIT THESE ===
INPUT_DIR        = 'PDFs'           # folder containing the PDFs
OUTPUT_BASE_DIR  = '.'               # parent folder where the dated output folder will be created

# Optional: override the 'today' reference for parsing relative dates like 'Yesterday' / 'Today'.
# Leave as None to pull from the page header timestamp inside the PDF (e.g. '5/15/26, 12:45 PM').
REFERENCE_DATE_OVERRIDE = None       # e.g. '2026-05-15'

## 2. Find PDFs

In [ ]:
from pathlib import Path

input_dir = Path(INPUT_DIR)
pdf_paths = sorted(input_dir.glob('*.pdf'))
if not pdf_paths:
    raise FileNotFoundError(f'No PDFs found in {input_dir.resolve()}')

print(f'Found {len(pdf_paths)} PDF(s):')
for p in pdf_paths:
    print(f'  - {p.name}')

## 3. Extract text + a reference date from each PDF

The Wealthsimple PDF stamps each page header with the export date in the form `M/D/YY, H:MM AM/PM`.
That gives us a deterministic anchor for `Yesterday` / `Today` headers.

In [ ]:
import pdfplumber, re
from datetime import datetime

PAGE_HEADER_TS = re.compile(r'(\d{1,2}/\d{1,2}/\d{2,4}),\s*\d{1,2}:\d{2}\s*[AP]M')

def extract_pdf(path):
    """Return (full_text, reference_datetime)."""
    chunks = []
    ref_dt = None
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            txt = page.extract_text() or ''
            chunks.append(txt)
            if ref_dt is None:
                m = PAGE_HEADER_TS.search(txt)
                if m:
                    for fmt in ('%m/%d/%y', '%m/%d/%Y'):
                        try:
                            ref_dt = datetime.strptime(m.group(1), fmt)
                            break
                        except ValueError:
                            continue
    return '\n'.join(chunks), ref_dt

if REFERENCE_DATE_OVERRIDE:
    global_ref = datetime.strptime(REFERENCE_DATE_OVERRIDE, '%Y-%m-%d')
else:
    global_ref = None

pdf_data = []
for p in pdf_paths:
    text, ref = extract_pdf(p)
    ref = global_ref or ref or datetime.now()
    pdf_data.append((p.name, text, ref))
    print(f'{p.name}: ref date = {ref.date()}, {len(text)} chars')

## 4. Parse transactions

PDF layout: a date header on its own line (`May 11, 2026`, `Yesterday`, `Today`), followed by
`Merchant\n− $X.XX CAD\nPurchase Credit card • …` blocks. We walk the text line-by-line, tracking the current
date, and emit `(date, merchant, amount)`. Credit card payments are skipped (not spend).

In [ ]:
from datetime import timedelta

DATE_HEADER = re.compile(r'^(YESTERDAY|TODAY|[A-Z][a-z]{2,8}\s+\d{1,2},\s*\d{4})\s*$', re.IGNORECASE)
AMOUNT_LINE = re.compile(r'^[−\-–—]\s*\$?([\d,]+\.\d{2})\s*CAD\s*$')
INLINE_TX   = re.compile(r'^(.+?)\s+[−\-–—]\s*\$?([\d,]+\.\d{2})\s*CAD\b')
NOISE_LINES = {
    'activity', 'scheduled activities', 'purchase credit card', 'pending',
    'from: chequing', 'this list only shows activity from april 1, 2023 onwards.',
    'to view more activity for cash, log in to the mobile app.',
}

def parse_label(label, ref_dt):
    label = label.strip()
    upper = label.upper()
    if upper == 'YESTERDAY':
        return (ref_dt - timedelta(days=1)).date().isoformat()
    if upper == 'TODAY':
        return ref_dt.date().isoformat()
    for fmt in ('%B %d, %Y', '%b %d, %Y'):
        try:
            return datetime.strptime(label, fmt).date().isoformat()
        except ValueError:
            continue
    return None

def is_noise(line):
    low = line.strip().lower()
    if not low:
        return True
    if low.startswith('purchase credit card'):
        return True
    if low.startswith('https://'):
        return True
    if PAGE_HEADER_TS.search(line):
        return True
    return low in NOISE_LINES

def parse_transactions(text, ref_dt):
    lines = text.splitlines()
    out = []
    current_date = None
    pending_merchant = None
    for raw in lines:
        line = raw.strip()
        if is_noise(line):
            pending_merchant = None
            continue
        m_date = DATE_HEADER.match(line)
        if m_date:
            d = parse_label(m_date.group(1), ref_dt)
            if d:
                current_date = d
            pending_merchant = None
            continue
        # Inline 'Merchant − $X.XX CAD' on a single line
        m_inline = INLINE_TX.match(line)
        if m_inline and current_date:
            merchant = m_inline.group(1).strip()
            if 'credit card payment' in merchant.lower():
                pending_merchant = None
                continue
            amount = float(m_inline.group(2).replace(',', ''))
            out.append((current_date, merchant, amount))
            pending_merchant = None
            continue
        # Amount on its own line — pair with the previous merchant line
        m_amt = AMOUNT_LINE.match(line)
        if m_amt and pending_merchant and current_date:
            amount = float(m_amt.group(1).replace(',', ''))
            if 'credit card payment' not in pending_merchant.lower():
                out.append((current_date, pending_merchant, amount))
            pending_merchant = None
            continue
        # Skip credit card payment headers entirely
        if 'credit card payment' in line.lower():
            pending_merchant = None
            continue
        # Otherwise this is probably the merchant name; remember it for the next amount line
        pending_merchant = line
    return out

all_tx = []
for name, text, ref in pdf_data:
    tx = parse_transactions(text, ref)
    print(f'{name}: parsed {len(tx)} transactions')
    all_tx.extend(tx)

# Dedup across PDFs by (date, merchant, amount)
seen = set()
transactions = []
for d, m, a in all_tx:
    key = (d, m.lower(), a)
    if key in seen:
        continue
    seen.add(key)
    transactions.append((d, m, a))

transactions.sort(key=lambda t: (t[0], -t[2], t[1]))
print(f'\nUnique transactions across all PDFs: {len(transactions)}')
print(f'Total spend: ${sum(t[2] for t in transactions):,.2f}')

## 5. Categorize

Lightweight rule-based categorizer — edit `CATEGORY_RULES` to teach it about your own merchants.

In [ ]:
CATEGORY_RULES = [
    ('Amazon',          ['amzn', 'amazon']),
    ('Professional',    ['engineers nova scotia', 'engineers ns', 'p.eng']),
    ('Subscriptions',   ['surfshark', 'koodo', 'apple.com', 'icloud', 'spotify', 'netflix',
                         'youtube', 'github', 'openai', 'anthropic', 'claude']),
    ('Restaurants',     ['poke', 'broth', 'ubereats', 'crepe', 'skip', 'doordash',
                         'tim hortons', 'starbucks', 'mcdonald']),
    ('Discount retail', ['dollarama', 'dollar tree', 'giant tiger']),
    ('Transportation',  ['communauto', 'uber canada', 'lyft', 'transit', 'parking',
                         'esso', 'shell', 'petro']),
    ('Home',            ['ikea', 'home depot', 'rona', 'canadian tire']),
    ('Groceries',       ['grocery', 'loblaws', 'sobeys', 'superstore', 'walmart',
                         'plaza', 'no frills']),
]

def categorize(merchant):
    m = merchant.lower()
    if 'uber canada' in m and 'ubereats' in m:
        return 'Restaurants'
    for cat, keywords in CATEGORY_RULES:
        if any(k in m for k in keywords):
            return cat
    return 'Other'

tx_with_cat = [(d, m, a, categorize(m)) for d, m, a in transactions]
for t in tx_with_cat[:8]:
    print(f'  {t[0]}  {t[1]:<40s}  ${t[2]:>8.2f}  [{t[3]}]')
print('  ...' if len(tx_with_cat) > 8 else '')

## 6. Create the dated output folder

In [ ]:
import pandas as pd

df = pd.DataFrame(tx_with_cat, columns=['date', 'merchant', 'amount', 'category'])

date_min = df.date.min()
date_max = df.date.max()
folder_name = f'{date_min}_to_{date_max}'
out_dir = Path(OUTPUT_BASE_DIR) / folder_name
out_dir.mkdir(parents=True, exist_ok=True)
print(f'Output folder: {out_dir.resolve()}')

## 7. Summary stats + `summary.md`

In [ ]:
total = df.amount.sum()
n_tx = len(df)
n_days = (pd.to_datetime(date_max) - pd.to_datetime(date_min)).days + 1
daily_avg = total / n_days
largest = df.loc[df.amount.idxmax()]
median_charge = df.amount.median()

by_cat = df.groupby('category').amount.agg(['sum', 'count']).sort_values('sum', ascending=False).round(2)
df['merchant_grp'] = df['merchant'].apply(
    lambda m: 'Amazon' if m.lower().startswith('amzn') else m
)
by_merchant = df.groupby('merchant_grp').amount.agg(['sum', 'count']).sort_values('sum', ascending=False).head(10).round(2)
by_day = df.groupby('date').amount.agg(['sum', 'count']).round(2)

# Write CSV
df[['date', 'merchant', 'amount', 'category']].to_csv(out_dir / 'transactions.csv', index=False)

# Write summary.md
lines = []
lines.append(f'# Credit card spending — {date_min} to {date_max}\n')
lines.append(f'**{n_days} days · {n_tx} transactions · ${total:,.2f} CAD total**\n')
lines.append(f'- Daily average: **${daily_avg:,.2f}**')
lines.append(f'- Median charge: **${median_charge:,.2f}**')
lines.append(f'- Largest charge: **${largest.amount:,.2f}** — {largest.merchant} ({largest.date})')
lines.append('')
lines.append('## By category')
lines.append('')
lines.append('| Category | Spend | # Tx | % |')
lines.append('|---|---:|---:|---:|')
for cat, row in by_cat.iterrows():
    pct = row['sum'] / total * 100
    lines.append(f"| {cat} | ${row['sum']:,.2f} | {int(row['count'])} | {pct:.1f}% |")
lines.append('')
lines.append('## Top merchants')
lines.append('')
lines.append('| Merchant | Spend | # Tx |')
lines.append('|---|---:|---:|')
for m, row in by_merchant.iterrows():
    lines.append(f"| {m} | ${row['sum']:,.2f} | {int(row['count'])} |")
lines.append('')
lines.append('## Daily')
lines.append('')
lines.append('| Date | Spend | # Tx |')
lines.append('|---|---:|---:|')
for d, row in by_day.iterrows():
    lines.append(f"| {d} | ${row['sum']:,.2f} | {int(row['count'])} |")

(out_dir / 'summary.md').write_text('\n'.join(lines), encoding='utf-8')
print(f"Wrote {out_dir / 'transactions.csv'}")
print(f"Wrote {out_dir / 'summary.md'}")
print()
print(by_cat)

## 8. Build the interactive dashboard HTML

Same Fraunces / JetBrains Mono / Inter look as the project's existing `spending_dashboard.html`.

In [ ]:
import json

CAT_COLORS = {
    'Amazon': '#d4a574', 'Professional': '#c4554f', 'Subscriptions': '#6ea8c4',
    'Restaurants': '#d97757', 'Discount retail': '#7cb992', 'Transportation': '#9a7ab8',
    'Home': '#b89968', 'Groceries': '#5e9b8a', 'Other': '#4a5468',
}

def _fmt(d, with_year=False):
    dt = datetime.strptime(d, '%Y-%m-%d')
    return f"{dt.strftime('%b')} {dt.day}" + (f", {dt.year}" if with_year else '')

date_range_label = f'{_fmt(date_min)} – {_fmt(date_max, with_year=True)}'

# Auto-build the 'what stood out' insights from the data itself.
def build_insights(df):
    insights = []
    # 1. Slow-leak category: highest-count category among small-ticket spends
    small = df[df.amount < 50]
    if len(small):
        leak = small.groupby('category').agg(
            count=('amount', 'size'), total=('amount', 'sum')
        ).sort_values('total', ascending=False)
        if len(leak) and leak.iloc[0]['count'] >= 4:
            cat = leak.index[0]
            insights.append((
                f'{cat} as a slow leak',
                f'<strong>{int(leak.iloc[0]["count"])} charges totalling ${leak.iloc[0]["total"]:,.2f}.</strong> '
                f'Most are under $50. Individually minor, collectively the largest small-ticket category. '
                f'A 24-hour wait rule typically trims this 20–30%.'
            ))
    # 2. Identical repeat charges
    repeats = df.groupby(['merchant', 'amount']).size().reset_index(name='n').sort_values('n', ascending=False)
    if len(repeats) and repeats.iloc[0]['n'] >= 3:
        r = repeats.iloc[0]
        monthly = r['amount'] * r['n'] * (30 / max(n_days, 1))
        insights.append((
            f'The ${r["amount"]:.2f} habit',
            f'{r["merchant"]} hit <strong>{int(r["n"])} times</strong> for the exact same ${r["amount"]:.2f}. '
            f'At this cadence that\'s roughly <strong>${monthly:,.0f}/month</strong> on a single merchant.'
        ))
    # 3. Frequency at one merchant
    by_m = df.groupby('merchant_grp').agg(n=('amount', 'size'), total=('amount', 'sum')).sort_values('n', ascending=False)
    by_m = by_m[by_m.index != 'Amazon']
    if len(by_m) and by_m.iloc[0]['n'] >= 4:
        name = by_m.index[0]
        insights.append((
            f'{name} drift',
            f'<strong>{int(by_m.iloc[0]["n"])} visits, ${by_m.iloc[0]["total"]:,.2f} total.</strong> '
            f'High-frequency small-store spending usually means forgotten items from the previous trip. '
            f'One consolidated weekly run typically cuts this 30–40%.'
        ))
    # 4. Subscriptions audit
    subs = df[df.category == 'Subscriptions']
    if len(subs):
        bullets = ', '.join(f'{r.merchant} ${r.amount:.2f}' for _, r in subs.head(4).iterrows())
        insights.append((
            'Subscriptions to audit',
            f'{bullets}. Confirm each is still in use and watch for double-billings inside the same week.'
        ))
    return insights

insights = build_insights(df)

tx_payload = [[r.date, r.merchant, float(r.amount), r.category] for _, r in df.iterrows()]
insights_payload = insights

# CSS / JS template ported from spending_dashboard.html
DASH_HTML = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Credit Card Spending — __RANGE__</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,400;9..144,500;9..144,600;9..144,700&family=JetBrains+Mono:wght@400;500;600&family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
  :root { --bg:#0a0e14; --panel:#11161f; --panel-2:#161c27; --ink:#e8ecf2; --ink-dim:#8a94a6; --ink-faint:#4a5468; --rule:#1f2632; --accent:#d4a574; --accent-2:#6ea8c4; --grid:rgba(138,148,166,0.08); }
  * { box-sizing:border-box; margin:0; padding:0; }
  html,body { background:var(--bg); color:var(--ink); font-family:'Inter',sans-serif; line-height:1.5; -webkit-font-smoothing:antialiased; }
  body { background:radial-gradient(ellipse 80% 50% at 10% 0%,rgba(212,165,116,0.06),transparent 60%),radial-gradient(ellipse 60% 40% at 90% 100%,rgba(110,168,196,0.05),transparent 60%),var(--bg); min-height:100vh; padding:48px 32px 80px; }
  .container { max-width:1280px; margin:0 auto; }
  header { margin-bottom:48px; border-bottom:1px solid var(--rule); padding-bottom:32px; }
  .eyebrow { font-family:'JetBrains Mono',monospace; font-size:11px; letter-spacing:0.18em; text-transform:uppercase; color:var(--accent); margin-bottom:12px; }
  h1 { font-family:'Fraunces',serif; font-weight:500; font-size:clamp(32px,4.5vw,56px); letter-spacing:-0.02em; line-height:1.05; margin-bottom:8px; }
  h1 em { font-style:italic; color:var(--accent); font-weight:400; }
  .subtitle { color:var(--ink-dim); font-size:15px; max-width:680px; }
  .kpis { display:grid; grid-template-columns:repeat(auto-fit,minmax(180px,1fr)); gap:1px; background:var(--rule); border:1px solid var(--rule); margin-bottom:32px; }
  .kpi { background:var(--panel); padding:24px 28px; }
  .kpi-label { font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.15em; text-transform:uppercase; color:var(--ink-dim); margin-bottom:12px; }
  .kpi-value { font-family:'Fraunces',serif; font-size:32px; font-weight:500; letter-spacing:-0.02em; line-height:1; }
  .kpi-sub { font-family:'JetBrains Mono',monospace; font-size:11px; color:var(--ink-faint); margin-top:8px; }
  .kpi-value.accent { color:var(--accent); }
  section { margin-bottom:48px; }
  .section-head { display:flex; align-items:baseline; justify-content:space-between; margin-bottom:20px; padding-bottom:12px; border-bottom:1px solid var(--rule); }
  .section-head h2 { font-family:'Fraunces',serif; font-weight:500; font-size:24px; letter-spacing:-0.01em; }
  .section-head .tag { font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.15em; color:var(--ink-faint); text-transform:uppercase; }
  .grid-3 { display:grid; grid-template-columns:2fr 1fr; gap:24px; }
  @media (max-width:880px) { .grid-3 { grid-template-columns:1fr; } }
  .card { background:var(--panel); border:1px solid var(--rule); padding:28px; }
  .card-title { font-family:'JetBrains Mono',monospace; font-size:11px; letter-spacing:0.15em; text-transform:uppercase; color:var(--ink-dim); margin-bottom:4px; }
  .card-headline { font-family:'Fraunces',serif; font-size:20px; font-weight:500; margin-bottom:24px; letter-spacing:-0.01em; }
  .chart-wrap { position:relative; height:320px; }
  .chart-wrap.tall { height:380px; }
  .cat-list { display:flex; flex-direction:column; gap:14px; }
  .cat-row { display:grid; grid-template-columns:140px 1fr 80px; gap:12px; align-items:center; }
  .cat-name { font-size:13px; color:var(--ink); font-weight:500; }
  .cat-bar-track { height:6px; background:rgba(138,148,166,0.1); position:relative; }
  .cat-bar-fill { height:100%; background:var(--accent); transition:width 0.6s cubic-bezier(.2,.7,.3,1); }
  .cat-amount { text-align:right; font-family:'JetBrains Mono',monospace; font-size:12px; color:var(--ink); }
  .cat-pct { font-family:'JetBrains Mono',monospace; font-size:10px; color:var(--ink-faint); }
  .insights { display:grid; grid-template-columns:repeat(auto-fit,minmax(280px,1fr)); gap:1px; background:var(--rule); border:1px solid var(--rule); }
  .insight { background:var(--panel); padding:24px 28px; }
  .insight-num { font-family:'Fraunces',serif; font-style:italic; font-size:14px; color:var(--accent); margin-bottom:8px; }
  .insight-title { font-family:'Fraunces',serif; font-size:18px; font-weight:500; margin-bottom:10px; letter-spacing:-0.01em; }
  .insight-body { font-size:13.5px; color:var(--ink-dim); line-height:1.6; }
  .insight-body strong { color:var(--ink); font-weight:500; }
  .table-wrap { background:var(--panel); border:1px solid var(--rule); overflow:hidden; }
  .table-controls { display:flex; gap:6px; padding:16px 20px; border-bottom:1px solid var(--rule); flex-wrap:wrap; }
  .chip { font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.1em; text-transform:uppercase; padding:6px 12px; background:transparent; border:1px solid var(--rule); color:var(--ink-dim); cursor:pointer; transition:all 0.15s; }
  .chip:hover { color:var(--ink); border-color:var(--ink-faint); }
  .chip.active { background:var(--accent); color:var(--bg); border-color:var(--accent); font-weight:600; }
  table { width:100%; border-collapse:collapse; }
  thead { font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.12em; text-transform:uppercase; color:var(--ink-faint); }
  th { text-align:left; padding:14px 20px; border-bottom:1px solid var(--rule); font-weight:500; }
  th.num { text-align:right; }
  tbody tr { border-bottom:1px solid rgba(31,38,50,0.5); transition:background 0.15s; }
  tbody tr:hover { background:var(--panel-2); }
  tbody tr.hidden { display:none; }
  td { padding:14px 20px; font-size:13px; }
  td.num { text-align:right; font-family:'JetBrains Mono',monospace; }
  td.date { font-family:'JetBrains Mono',monospace; color:var(--ink-dim); font-size:12px; }
  .cat-tag { display:inline-block; font-family:'JetBrains Mono',monospace; font-size:9.5px; letter-spacing:0.1em; text-transform:uppercase; padding:3px 8px; border:1px solid var(--rule); color:var(--ink-dim); }
  footer { margin-top:64px; padding-top:24px; border-top:1px solid var(--rule); font-family:'JetBrains Mono',monospace; font-size:11px; color:var(--ink-faint); display:flex; justify-content:space-between; flex-wrap:wrap; gap:12px; }
  .links { font-family:'JetBrains Mono',monospace; font-size:11px; margin-top:16px; }
  .links a { color:var(--accent-2); text-decoration:none; margin-right:18px; }
  .links a:hover { text-decoration:underline; }
</style>
</head>
<body>
<div class="container">
  <header>
    <div class="eyebrow">Wealthsimple Credit Card · __RANGE__</div>
    <h1>__NDAYS__ days, <em>__NTX__ charges</em>.</h1>
    <p class="subtitle">A close read of where the money actually went, what's recurring, and which categories quietly compound. Filterable by category below.</p>
    <div class="links"><a href="suggestions.html">→ See suggestions</a><a href="summary.md">→ summary.md</a><a href="transactions.csv">→ transactions.csv</a></div>
  </header>
  <div class="kpis">
    <div class="kpi"><div class="kpi-label">Total Spend</div><div class="kpi-value accent">$__TOTAL__</div><div class="kpi-sub">CAD · __NTX__ transactions</div></div>
    <div class="kpi"><div class="kpi-label">Daily Avg</div><div class="kpi-value">$__AVG__</div><div class="kpi-sub">across __NDAYS__ days</div></div>
    <div class="kpi"><div class="kpi-label">Largest Charge</div><div class="kpi-value">$__LARGEST__</div><div class="kpi-sub">__LARGEST_LABEL__</div></div>
    <div class="kpi"><div class="kpi-label">Median Charge</div><div class="kpi-value">$__MEDIAN__</div><div class="kpi-sub">half above, half below</div></div>
  </div>
  <section><div class="section-head"><h2>Cumulative spend</h2><span class="tag">Running total</span></div>
    <div class="card"><div class="card-title">Where the curve steepens</div><div class="card-headline">__CUM_HEADLINE__</div><div class="chart-wrap tall"><canvas id="cumChart"></canvas></div></div></section>
  <section class="grid-3">
    <div class="card"><div class="card-title">Daily activity</div><div class="card-headline">__DAILY_HEADLINE__</div><div class="chart-wrap"><canvas id="dailyChart"></canvas></div></div>
    <div class="card"><div class="card-title">Category share</div><div class="card-headline">__CAT_HEADLINE__</div><div class="chart-wrap"><canvas id="catDonut"></canvas></div></div>
  </section>
  <section style="margin-top:48px;"><div class="section-head"><h2>By category</h2><span class="tag">Sorted by spend</span></div><div class="card"><div class="cat-list" id="catList"></div></div></section>
  <section><div class="section-head"><h2>Top merchants</h2><span class="tag">Where it really goes</span></div><div class="card"><div class="chart-wrap tall"><canvas id="merchChart"></canvas></div></div></section>
  <section><div class="section-head"><h2>What stood out</h2><span class="tag">Auto-detected patterns</span></div><div class="insights" id="insightsBox"></div></section>
  <section><div class="section-head"><h2>All transactions</h2><span class="tag">Tap a category to filter</span></div>
    <div class="table-wrap"><div class="table-controls" id="filters"></div>
      <table><thead><tr><th style="width:90px;">Date</th><th>Merchant</th><th>Category</th><th class="num">Amount</th></tr></thead><tbody id="txBody"></tbody></table></div></section>
  <footer><span>Source: Wealthsimple credit card PDF activity export</span><span>Generated from PDFs</span></footer>
</div>
<script>
const TX = __TX_JSON__;
const INSIGHTS = __INSIGHTS_JSON__;
const CAT_COLORS = __CAT_COLORS__;
const INK="#e8ecf2", DIM="#8a94a6", GRID="rgba(138,148,166,0.08)";
const FONT="'JetBrains Mono', monospace";
Chart.defaults.color=DIM; Chart.defaults.font.family="Inter, sans-serif"; Chart.defaults.font.size=11;
const sorted=[...TX].sort((a,b)=>a[0].localeCompare(b[0]));
const dailyMap={}; sorted.forEach(t=>{dailyMap[t[0]]=(dailyMap[t[0]]||0)+t[2];});
const days=Object.keys(dailyMap).sort(); let running=0; const cum=days.map(d=>{running+=dailyMap[d]; return running;});
new Chart(document.getElementById("cumChart"),{type:"line",data:{labels:days.map(d=>d.slice(5)),datasets:[{data:cum,borderColor:"#d4a574",backgroundColor:ctx=>{const c=ctx.chart.ctx.createLinearGradient(0,0,0,380);c.addColorStop(0,"rgba(212,165,116,0.25)");c.addColorStop(1,"rgba(212,165,116,0)");return c;},fill:true,tension:0.3,borderWidth:2,pointRadius:4,pointBackgroundColor:"#d4a574",pointBorderColor:"#0a0e14",pointBorderWidth:2}]},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{backgroundColor:"#161c27",borderColor:"#1f2632",borderWidth:1,titleFont:{family:FONT,size:10},bodyFont:{family:FONT,size:12},padding:12,titleColor:DIM,bodyColor:INK,callbacks:{label:c=>"  $"+c.parsed.y.toFixed(2)}}},scales:{x:{grid:{color:GRID},ticks:{font:{family:FONT,size:10}}},y:{grid:{color:GRID},ticks:{font:{family:FONT,size:10},callback:v=>"$"+v}}}}});
new Chart(document.getElementById("dailyChart"),{type:"bar",data:{labels:days.map(d=>d.slice(5)),datasets:[{data:days.map(d=>dailyMap[d]),backgroundColor:"#6ea8c4",borderRadius:0}]},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{backgroundColor:"#161c27",borderColor:"#1f2632",borderWidth:1,titleFont:{family:FONT,size:10},bodyFont:{family:FONT,size:12},padding:12,titleColor:DIM,bodyColor:INK,callbacks:{label:c=>"  $"+c.parsed.y.toFixed(2)}}},scales:{x:{grid:{color:GRID},ticks:{font:{family:FONT,size:10}}},y:{grid:{color:GRID},ticks:{font:{family:FONT,size:10},callback:v=>"$"+v}}}}});
const catTotals={}; TX.forEach(t=>{catTotals[t[3]]=(catTotals[t[3]]||0)+t[2];});
const catEntries=Object.entries(catTotals).sort((a,b)=>b[1]-a[1]);
const totalAll=catEntries.reduce((s,c)=>s+c[1],0);
new Chart(document.getElementById("catDonut"),{type:"doughnut",data:{labels:catEntries.map(c=>c[0]),datasets:[{data:catEntries.map(c=>c[1]),backgroundColor:catEntries.map(c=>CAT_COLORS[c[0]]||"#4a5468"),borderColor:"#11161f",borderWidth:2}]},options:{responsive:true,maintainAspectRatio:false,cutout:"62%",plugins:{legend:{position:"right",labels:{font:{family:"Inter",size:11},color:DIM,padding:10,boxWidth:10,boxHeight:10}},tooltip:{backgroundColor:"#161c27",borderColor:"#1f2632",borderWidth:1,titleFont:{family:FONT,size:10},bodyFont:{family:FONT,size:12},padding:12,titleColor:DIM,bodyColor:INK,callbacks:{label:c=>"  $"+c.parsed.toFixed(2)+" · "+((c.parsed/totalAll)*100).toFixed(1)+"%"}}}}});
const maxCat=catEntries[0][1]; const catList=document.getElementById("catList");
catEntries.forEach(([name,amt])=>{const pct=(amt/totalAll)*100; const w=(amt/maxCat)*100; const row=document.createElement("div"); row.className="cat-row"; row.innerHTML=`<span class="cat-name">${name}</span><div><div class="cat-bar-track"><div class="cat-bar-fill" style="width:${w}%; background:${CAT_COLORS[name]||"#4a5468"};"></div></div><span class="cat-pct">${pct.toFixed(1)}% of total</span></div><span class="cat-amount">$${amt.toFixed(2)}</span>`; catList.appendChild(row);});
const merchTotals={}; TX.forEach(t=>{const key=t[1].toLowerCase().startsWith("amzn")?"Amazon":(t[1].toLowerCase().includes("poke go")?"Poke Go & Broth House":t[1]); merchTotals[key]=(merchTotals[key]||0)+t[2];});
const merchEntries=Object.entries(merchTotals).sort((a,b)=>b[1]-a[1]).slice(0,10);
new Chart(document.getElementById("merchChart"),{type:"bar",data:{labels:merchEntries.map(m=>m[0]),datasets:[{data:merchEntries.map(m=>m[1]),backgroundColor:"#d4a574",borderRadius:0}]},options:{indexAxis:"y",responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{backgroundColor:"#161c27",borderColor:"#1f2632",borderWidth:1,titleFont:{family:FONT,size:10},bodyFont:{family:FONT,size:12},padding:12,titleColor:DIM,bodyColor:INK,callbacks:{label:c=>"  $"+c.parsed.x.toFixed(2)}}},scales:{x:{grid:{color:GRID},ticks:{font:{family:FONT,size:10},callback:v=>"$"+v}},y:{grid:{display:false},ticks:{font:{family:"Inter",size:12},color:INK}}}}});
const insightsBox=document.getElementById("insightsBox");
INSIGHTS.forEach(([title,body],i)=>{const d=document.createElement("div");d.className="insight";d.innerHTML=`<div class="insight-num">${String(i+1).padStart(2,"0")}</div><div class="insight-title">${title}</div><div class="insight-body">${body}</div>`;insightsBox.appendChild(d);});
if(INSIGHTS.length===0){insightsBox.innerHTML='<div class="insight"><div class="insight-body">No notable patterns detected for this period.</div></div>';}
const filtersEl=document.getElementById("filters"); const bodyEl=document.getElementById("txBody");
const cats=["All",...catEntries.map(c=>c[0])];
cats.forEach(c=>{const b=document.createElement("button"); b.className="chip"+(c==="All"?" active":""); b.textContent=c; b.onclick=()=>{document.querySelectorAll(".chip").forEach(x=>x.classList.remove("active")); b.classList.add("active"); document.querySelectorAll("#txBody tr").forEach(r=>{r.classList.toggle("hidden",c!=="All" && r.dataset.cat!==c);});}; filtersEl.appendChild(b);});
const txSorted=[...TX].sort((a,b)=>b[0].localeCompare(a[0])||b[2]-a[2]);
txSorted.forEach(t=>{const tr=document.createElement("tr"); tr.dataset.cat=t[3]; const color=CAT_COLORS[t[3]]||"#4a5468"; tr.innerHTML=`<td class="date">${t[0].slice(5)}</td><td>${t[1]}</td><td><span class="cat-tag" style="color:${color}; border-color:${color}33;">${t[3]}</span></td><td class="num">$${t[2].toFixed(2)}</td>`; bodyEl.appendChild(tr);});
</script></body></html>'''

# Headline strings — small data-driven blurbs
top_cat = by_cat.index[0]
top_cat_pct = by_cat.iloc[0]['sum'] / total * 100
top_day = by_day['sum'].idxmax()
top_day_amt = by_day['sum'].max()
cum_headline = 'Spend distributed across the period.' if df.amount.std() < df.amount.mean() else 'A handful of days drove most of the total.'
daily_headline = f'${top_day_amt:,.0f} on {top_day} was the biggest day.'
cat_headline = f'{top_cat} alone is {top_cat_pct:.0f}% of spend.'

html = (DASH_HTML
    .replace('__RANGE__', date_range_label)
    .replace('__NDAYS__', str(n_days))
    .replace('__NTX__', str(n_tx))
    .replace('__TOTAL__', f'{total:,.2f}')
    .replace('__AVG__', f'{daily_avg:,.2f}')
    .replace('__LARGEST__', f'{largest.amount:,.2f}')
    .replace('__LARGEST_LABEL__', f'{largest.merchant} · {largest.date[5:]}')
    .replace('__MEDIAN__', f'{median_charge:,.2f}')
    .replace('__CUM_HEADLINE__', cum_headline)
    .replace('__DAILY_HEADLINE__', daily_headline)
    .replace('__CAT_HEADLINE__', cat_headline)
    .replace('__TX_JSON__', json.dumps(tx_payload))
    .replace('__INSIGHTS_JSON__', json.dumps(insights_payload))
    .replace('__CAT_COLORS__', json.dumps(CAT_COLORS))
)

(out_dir / 'spending_dashboard.html').write_text(html, encoding='utf-8')
print(f"Wrote {out_dir / 'spending_dashboard.html'}")

## 9. Build the `suggestions.html` page

Rule-based recommendations engine. Each suggestion has a title, a body explaining what's happening,
and an **estimated monthly savings** if you act on it. Edit `build_suggestions(df)` to tune the rules.

In [ ]:
def build_suggestions(df, n_days):
    """Return a list of dicts: {title, body, est_savings_month, impact}."""
    monthly_factor = 30 / max(n_days, 1)
    s = []

    # 1. Identical recurring charges (likely habits/subscriptions you can prune)
    rep = df.groupby(['merchant', 'amount']).size().reset_index(name='n').sort_values('n', ascending=False)
    for _, r in rep.iterrows():
        if r['n'] < 3:
            break
        period_spend = r['amount'] * r['n']
        monthly = period_spend * monthly_factor
        s.append({
            'title': f'Cut one {r["merchant"]} order per week',
            'body': (f'You spent <strong>${r["amount"]:.2f} × {int(r["n"])} = ${period_spend:.2f}</strong> '
                     f'on {r["merchant"]} alone over {n_days} days. Dropping just one of these per week '
                     f'cuts about a quarter of the habit cost.'),
            'est_savings_month': round(monthly * 0.25, 2),
            'impact': 'medium',
        })

    # 2. Amazon clustering (orders < 7 days apart)
    amzn = df[df.category == 'Amazon'].copy()
    if len(amzn) >= 4:
        amzn_total = amzn.amount.sum()
        monthly = amzn_total * monthly_factor
        s.append({
            'title': 'Consolidate Amazon orders with a 24-hour wait rule',
            'body': (f'<strong>{len(amzn)} Amazon orders totalling ${amzn_total:,.2f}</strong> in {n_days} days. '
                     f'Most are small impulse buys. A 24-hour cart wait and a once-weekly checkout '
                     f'typically trims 15–20% of Amazon spend.'),
            'est_savings_month': round(monthly * 0.175, 2),
            'impact': 'high' if amzn_total > 200 else 'medium',
        })

    # 3. Small-store frequency drift
    small = df[df.amount < 50]
    by_m = small.groupby('merchant_grp').agg(n=('amount', 'size'), total=('amount', 'sum')).sort_values('n', ascending=False)
    by_m = by_m[by_m.index != 'Amazon']
    for name, row in by_m.iterrows():
        if row['n'] < 4:
            break
        monthly = row['total'] * monthly_factor
        s.append({
            'title': f'Consolidate {name} visits into one weekly run',
            'body': (f'<strong>{int(row["n"])} visits, ${row["total"]:,.2f}</strong> at {name}. '
                     f'High-frequency stops at one store usually mean you\'re returning for forgotten items. '
                     f'A single weekly list-driven trip typically cuts this 30–40%.'),
            'est_savings_month': round(monthly * 0.35, 2),
            'impact': 'medium',
        })

    # 4. Subscriptions audit
    subs = df[df.category == 'Subscriptions']
    if len(subs):
        subs_total = subs.amount.sum()
        items = ', '.join(f'<strong>{r.merchant}</strong> (${r.amount:.2f})' for _, r in subs.iterrows())
        monthly = subs_total * monthly_factor
        s.append({
            'title': 'Audit subscriptions — cancel one',
            'body': (f'Active in this window: {items}. Pick the one you used least this month and cancel. '
                     f'A single dropped subscription is the cleanest recurring win.'),
            'est_savings_month': round(monthly / max(len(subs), 1), 2),
            'impact': 'high',
        })

    # 5. Doubled-up charges from the same merchant in <7 days
    df_d = df.copy()
    df_d['dt'] = pd.to_datetime(df_d['date'])
    for merchant, grp in df_d.groupby('merchant'):
        if len(grp) < 2:
            continue
        grp = grp.sort_values('dt')
        gaps = grp['dt'].diff().dt.days
        close = (gaps <= 3).sum()
        if close >= 1 and 'apple.com' in merchant.lower():
            s.append({
                'title': f'Verify {merchant} double-billing',
                'body': (f'<strong>{merchant} billed {len(grp)} times</strong> in this window, '
                         f'including charges within 3 days of each other. '
                         f'Worth confirming iCloud storage vs. an in-app purchase you forgot about.'),
                'est_savings_month': round(grp.amount.min() * monthly_factor, 2),
                'impact': 'low',
            })
            break

    # 6. Daily pace check
    daily_avg = df.amount.sum() / n_days
    if daily_avg > 60:
        s.append({
            'title': 'Set a $50/day cap and watch the slope',
            'body': (f'Daily average right now is <strong>${daily_avg:,.2f}</strong>. Sliding that to $50 '
                     f'would save roughly <strong>${(daily_avg - 50) * 30:,.0f}/month</strong>. The cumulative-spend '
                     f'chart in the dashboard is the fastest way to spot the days that blew past it.'),
            'est_savings_month': round((daily_avg - 50) * 30, 2),
            'impact': 'high',
        })

    # Sort: highest impact + savings first
    impact_rank = {'high': 0, 'medium': 1, 'low': 2}
    s.sort(key=lambda x: (impact_rank.get(x['impact'], 3), -x['est_savings_month']))
    return s

suggestions = build_suggestions(df, n_days)
total_potential = sum(x['est_savings_month'] for x in suggestions)
print(f'Built {len(suggestions)} suggestions. Estimated combined monthly savings: ${total_potential:,.2f}')
for x in suggestions:
    print(f"  [{x['impact']:6s}] ${x['est_savings_month']:>7,.2f}/mo — {x['title']}")

In [ ]:
SUGG_HTML = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Suggestions — __RANGE__</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,400;9..144,500;9..144,600&family=JetBrains+Mono:wght@400;500;600&family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
<style>
  :root { --bg:#0a0e14; --panel:#11161f; --panel-2:#161c27; --ink:#e8ecf2; --ink-dim:#8a94a6; --ink-faint:#4a5468; --rule:#1f2632; --accent:#d4a574; --accent-2:#6ea8c4; --high:#c4554f; --med:#d4a574; --low:#6ea8c4; }
  * { box-sizing:border-box; margin:0; padding:0; }
  html,body { background:var(--bg); color:var(--ink); font-family:'Inter',sans-serif; line-height:1.5; -webkit-font-smoothing:antialiased; }
  body { background:radial-gradient(ellipse 80% 50% at 10% 0%,rgba(212,165,116,0.06),transparent 60%),radial-gradient(ellipse 60% 40% at 90% 100%,rgba(110,168,196,0.05),transparent 60%),var(--bg); min-height:100vh; padding:48px 32px 80px; }
  .container { max-width:920px; margin:0 auto; }
  header { margin-bottom:48px; border-bottom:1px solid var(--rule); padding-bottom:32px; }
  .eyebrow { font-family:'JetBrains Mono',monospace; font-size:11px; letter-spacing:0.18em; text-transform:uppercase; color:var(--accent); margin-bottom:12px; }
  h1 { font-family:'Fraunces',serif; font-weight:500; font-size:clamp(32px,4.5vw,52px); letter-spacing:-0.02em; line-height:1.05; margin-bottom:8px; }
  h1 em { font-style:italic; color:var(--accent); font-weight:400; }
  .subtitle { color:var(--ink-dim); font-size:15px; max-width:680px; }
  .links { font-family:'JetBrains Mono',monospace; font-size:11px; margin-top:16px; }
  .links a { color:var(--accent-2); text-decoration:none; margin-right:18px; }
  .links a:hover { text-decoration:underline; }
  .total-bar { background:var(--panel); border:1px solid var(--rule); padding:24px 28px; margin-bottom:32px; display:flex; align-items:baseline; justify-content:space-between; flex-wrap:wrap; gap:12px; }
  .total-label { font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.18em; text-transform:uppercase; color:var(--ink-dim); }
  .total-value { font-family:'Fraunces',serif; font-size:40px; font-weight:500; letter-spacing:-0.02em; color:var(--accent); }
  .total-sub { font-family:'JetBrains Mono',monospace; font-size:11px; color:var(--ink-faint); }
  .sugg { background:var(--panel); border:1px solid var(--rule); border-left:3px solid var(--accent); padding:28px 32px; margin-bottom:16px; display:grid; grid-template-columns:1fr 160px; gap:24px; align-items:start; }
  @media (max-width:680px) { .sugg { grid-template-columns:1fr; } }
  .sugg.impact-high { border-left-color:var(--high); }
  .sugg.impact-medium { border-left-color:var(--med); }
  .sugg.impact-low { border-left-color:var(--low); }
  .sugg-num { font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.18em; color:var(--ink-faint); margin-bottom:8px; }
  .sugg-title { font-family:'Fraunces',serif; font-size:22px; font-weight:500; letter-spacing:-0.01em; margin-bottom:12px; }
  .sugg-body { font-size:14px; color:var(--ink-dim); line-height:1.65; }
  .sugg-body strong { color:var(--ink); font-weight:500; }
  .sugg-meta { text-align:right; }
  .sugg-amt { font-family:'Fraunces',serif; font-size:28px; font-weight:500; color:var(--ink); line-height:1; }
  .sugg-amt-label { font-family:'JetBrains Mono',monospace; font-size:9.5px; letter-spacing:0.15em; text-transform:uppercase; color:var(--ink-faint); margin-top:6px; }
  .sugg-impact { display:inline-block; margin-top:14px; font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.15em; padding:4px 10px; border:1px solid var(--rule); text-transform:uppercase; }
  .impact-high .sugg-impact { color:var(--high); border-color:var(--high); }
  .impact-medium .sugg-impact { color:var(--med); border-color:var(--med); }
  .impact-low .sugg-impact { color:var(--low); border-color:var(--low); }
  footer { margin-top:64px; padding-top:24px; border-top:1px solid var(--rule); font-family:'JetBrains Mono',monospace; font-size:11px; color:var(--ink-faint); }
  .empty { background:var(--panel); border:1px solid var(--rule); padding:48px 32px; text-align:center; color:var(--ink-dim); }
</style>
</head>
<body>
<div class="container">
  <header>
    <div class="eyebrow">Suggestions · __RANGE__</div>
    <h1>Where you could <em>actually</em> save.</h1>
    <p class="subtitle">Rule-based reads of the patterns in this period. Every estimate assumes you act on the suggestion; ignoring it costs you the same amount.</p>
    <div class="links"><a href="spending_dashboard.html">← back to dashboard</a><a href="summary.md">summary.md</a></div>
  </header>
  <div class="total-bar">
    <div><div class="total-label">Combined potential</div><div class="total-value">$__TOTAL_POT__</div></div>
    <div class="total-sub">per month, if you act on all of the below</div>
  </div>
  __SUGGESTIONS__
  <footer>Generated automatically from your PDF activity export.</footer>
</div>
</body></html>'''

if suggestions:
    blocks = []
    for i, x in enumerate(suggestions, 1):
        blocks.append(f'''    <div class="sugg impact-{x["impact"]}">
      <div>
        <div class="sugg-num">{i:02d}</div>
        <div class="sugg-title">{x["title"]}</div>
        <div class="sugg-body">{x["body"]}</div>
      </div>
      <div class="sugg-meta">
        <div class="sugg-amt">${x["est_savings_month"]:,.0f}</div>
        <div class="sugg-amt-label">est. /month</div>
        <div class="sugg-impact">{x["impact"]} impact</div>
      </div>
    </div>''')
    body = '\n'.join(blocks)
else:
    body = '    <div class="empty">No notable patterns to suggest on for this period.</div>'

sugg_html = (SUGG_HTML
    .replace('__RANGE__', date_range_label)
    .replace('__TOTAL_POT__', f'{total_potential:,.0f}')
    .replace('__SUGGESTIONS__', body)
)

(out_dir / 'suggestions.html').write_text(sugg_html, encoding='utf-8')
print(f"Wrote {out_dir / 'suggestions.html'}")

## 10. Done

Open the dashboard in your browser — `suggestions.html` is linked in the header.

In [ ]:
print('All outputs:')
for f in sorted(out_dir.iterdir()):
    print(f'  - {f}')
print()
print(f'Open: {(out_dir / "spending_dashboard.html").resolve()}')

## Tuning tips

- **Wrong dates from `Yesterday`/`Today`** → set `REFERENCE_DATE_OVERRIDE = '2026-05-15'` in cell 2.
- **Missing or misparsed transactions** → print the raw text for the offending PDF (`pdf_data[i][1]`) and
  adjust the `DATE_HEADER` / `AMOUNT_LINE` / `INLINE_TX` regexes in the parser cell.
- **Everything ends up `Other`** → add merchant keywords to `CATEGORY_RULES`.
- **New bank's PDF layout** → look at the raw text first, then tweak the parser. The downstream cells
  (summary, dashboard, suggestions) are layout-agnostic once you have a `df` with
  `date, merchant, amount, category` columns.